In [1]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px


In [2]:
img = cv2.imread('data/image2_base.png')
mask = cv2.imread('data/mask2.png', 0)

#invertion masque
mask = cv2.bitwise_not(mask)


# 2. On crée un "stack" (un paquet) des deux images
# Pour que ça marche, le masque doit avoir 3 canaux comme l'image
mask_3ch = cv2.merge([mask, mask, mask])
combined = np.stack([cv2.cvtColor(img, cv2.COLOR_BGR2RGB), mask_3ch])

# 3. Affichage avec Plotly Express
# facet_col=0 dit à Plotly de séparer le "stack" en deux colonnes
fig = px.imshow(combined, facet_col=0, binary_string=True)

# 4. On nomme les axes et les titres
fig.layout.annotations[0].text = "Image Originale"
fig.layout.annotations[1].text = "Masque"

fig.update_layout(showlegend=False)
fig.show()



In [3]:
# pix_mask =[]
# for i in range(len(mask)) :
#     for j in range(len(mask[0])):
#         if mask[i,j] < 100 :
#             pix_mask.append([i,j])
# print(pix_mask)

mask = cv2.resize(mask, (img.shape[1], img.shape[0]))  # d'abord

pix_mask = []
for i in range(mask.shape[0]):
    for j in range(mask.shape[1]):
        if mask[i,j] < 100:
            pix_mask.append([i,j])

mask = cv2.resize(mask, (img.shape[1], img.shape[0]))
# mask_bin = cv2.threshold(mask, 127, 255, cv2.THRESH_BINARY)


# 3. Récupération du masque binaire (on prend l'index [1])
_, mask_only = cv2.threshold(mask, 127, 255, cv2.THRESH_BINARY)

# 4. Application du masque
# On fait un "ET" entre l'image et elle-même, mais UNIQUEMENT là où le masque est blanc
img_m = cv2.bitwise_and(img, img, mask=mask_only)

# 5. Affichage Plotly Express
# Conversion BGR -> RGB pour que les couleurs soient les bonnes
img_m_rgb = cv2.cvtColor(img_m, cv2.COLOR_BGR2RGB)
fig = px.imshow(img_m_rgb)
fig.update_layout(title="Aperçu de la zone masquée (Coordonnées actives)")
fig.show()




In [4]:
#algo

#noyeau 3 3
k = np.array([[1, 1, 1],
              [1, 0, 1],
              [1, 1, 1]])/8


k_5x5 = np.array([
    [1, 2,  4, 2, 1],
    [2, 4,  8, 4, 2],
    [4, 8,  0, 8, 4],  
    [2, 4,  8, 4, 2],
    [1, 2,  4, 2, 1]
])  
k_5x5 = k_5x5 / k_5x5.sum()  # normalisation propre



# def algo_oliveira(img, mask_indices):
#     image = img.astype(np.float32)
#     # On crée un masque binaire (1 pour image saine, 0 pour trou)
#     mask_map = np.ones(image.shape[:2], dtype=np.float32)
#     for y, x in mask_indices:
#         mask_map[y, x] = 0

#     for i in mask_indices:
#         y, x = i[0], i[1]
        
#         if 2 <= y < image.shape[0] - 2 and 2 <= x < image.shape[1] - 2:
#             # 1. On extrait les voisins de l'image ET du masque binaire
#             voisins = image[y-2:y+3, x-2:x+3]
#             voisins_valides = mask_map[y-2:y+3, x-2:x+3] # 1 si sain, 0 si masque
            
#             # 2. On ajuste le noyau : on ne garde que les poids des pixels SAINS
#             # On multiplie le noyau par le masque de validité
#             noyau_adapte = k_5x5 * voisins_valides
            
#             # 3. Normalisation CRUCIALE : 
#             # La somme des poids doit toujours être 1, mais seulement sur les pixels valides
#             somme_poids = np.sum(noyau_adapte)
            
#             if somme_poids > 0:
#                 noyau_normalise = noyau_adapte / somme_poids
#                 for c in range(3):
#                     image[y, x, c] = np.sum(voisins[:, :, c] * noyau_normalise)
    
#     return image.astype(np.uint8)






mask_bool = np.zeros(img_m_rgb.shape[:2], dtype=bool)
for y, x in pix_mask:
    mask_bool[y, x] = True

k_5x5 = np.array([
    [1, 2,  4, 2, 1],
    [2, 4,  8, 4, 2],
    [4, 8,  0, 8, 4], 
    [2, 4,  8, 4, 2],
    [1, 2,  4, 2, 1]
], dtype=np.float32) / 100

In [ ]:
def priorité(patch_grad_x, patch_grad_y, patch_C, normale_p, R):
    # C(p)
    surface = (2 * R + 1) ** 2
    confiance = np.sum(patch_C) / surface

    # D(p) gradient au pixel central p
    gx = patch_grad_x[R, R]
    gy = patch_grad_y[R, R]
    grad_perp = np.array([-gy, gx])
    donnee = abs(np.dot(grad_perp, normale_p)) / 255.0

    return confiance * donnee, confiance

In [6]:
mask = cv2.bitwise_not(mask)
#affichage mask
px.imshow(mask).update_layout(title="Masque inversé").show()

In [ ]:




def algo_criminisi(image, mask, largeur_patch):
    R = (largeur_patch - 1) // 2
    h_orig, w_orig = image.shape[:2]
    
    #padding pour eviter les probleme bord si mask est proche du bord exemple img saut de l'ange
    image = cv2.copyMakeBorder(image, R, R, R, R, cv2.BORDER_REFLECT)
    mask = cv2.copyMakeBorder(mask, R, R, R, R, cv2.BORDER_CONSTANT, value=0)
    
    C = np.where(mask == 0, 1.0, 0.0)
    

    # gros_noyau = np.ones((2 * largeur_patch - 1, 2 * largeur_patch - 1), np.uint8)

    gros_noyau = np.ones((largeur_patch, largeur_patch), np.uint8)
    
    
    #dimension image
    h, w = image.shape[:2] 
    marge = largeur_patch -1
    iter_count = 0
    while np.any(mask > 0):
        #compteur iterations
       
        iter_count += 1
        print(f"Pixels restants : {np.sum(mask > 0)}")
        
        #maj grad image et normales
        gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY).astype(np.float32)
        grad_x_img = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
        grad_y_img = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)
        
        
        mask_normal = C.astype(np.float32)
        nx_img = cv2.Sobel(mask_normal, cv2.CV_64F, 1, 0, ksize=3)
        ny_img = cv2.Sobel(mask_normal, cv2.CV_64F, 0, 1, ksize=3)
        
        
        zone_interdite = cv2.dilate(mask, gros_noyau, iterations=1)
        
        
        
        
        frontiere, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
        points_frontiere = np.vstack(frontiere)
        liste_pixels_frontiere = points_frontiere[:, 0]
       
       #verif
        print(f"Nombre de contours : {len(frontiere)}")
        for i, c in enumerate(frontiere):
            print(f"  Contour {i} : {len(c)} points, premier point : {c[0]}")
       
       
       #init
        pix_coord = None
        prio = -1
        
        #priotité
        for x, y in liste_pixels_frontiere:
                    
            ymin = y - R
            ymax = y + R + 1
            xmin = x - R
            xmax = x + R + 1
         
         
            if not (ymin < 0 or ymax > h or xmin < 0 or xmax > w):
    
                
                nx = nx_img[y, x]
                ny = ny_img[y, x]
                norme = np.hypot(nx, ny)
                normale_mask = np.array([nx, ny]) / norme if norme > 0 else np.array([0.0, 0.0])
                    

                patch_grad_x = grad_x_img[y-R : y+R+1, x-R : x+R+1]
                patch_grad_y = grad_y_img[y-R : y+R+1, x-R : x+R+1]
                patch_confiance = C[ymin:ymax, xmin:xmax]
                
                new_prio, new_C_p = priorité(patch_grad_x, patch_grad_y, patch_confiance, normale_mask, R)
                if new_prio > prio:
                    prio = new_prio
                    C_p = new_C_p
                    pix_coord = [x, y]
    
            
        if pix_coord is None:
            print("Aucun pixel valide")
            break
        
       # Extraction patch cible
        x, y = pix_coord[0], pix_coord[1]
        ymin, ymax = y - R, y + R + 1
        xmin, xmax = x - R, x + R + 1

        patch_cible        = image[ymin:ymax, xmin:xmax]
        patch_mask_region  = mask[ymin:ymax, xmin:xmax]
        indices_patch_mask = (patch_mask_region > 0)   # pixels à remplir
        pixels_connus      = ~indices_patch_mask        # pixels utilisables

        meilleur_score = float('inf')
        meilleur_patch = None

        # a opti , 40min pour reslutat img2
        for py in range(marge, h - marge):
            for px in range(marge, w - marge):

                patch_mask_corr = mask[py-R:py+R+1, px-R:px+R+1]
                if np.any(patch_mask_corr > 0):
                    continue

                patch_corr = image[py - R : py + R + 1, px - R : px + R + 1]

                patch_cible_lab = cv2.cvtColor(patch_cible, cv2.COLOR_RGB2Lab).astype(np.float32) #doc : We use the CIE Lab colour space because of its property of perceptual uniformity
                patch_corr_lab  = cv2.cvtColor(patch_corr,  cv2.COLOR_RGB2Lab).astype(np.float32)
                score = np.sum((patch_cible_lab[pixels_connus] - patch_corr_lab[pixels_connus]) ** 2)

                if score < meilleur_score:
                    meilleur_score = score
                    meilleur_patch = patch_corr.copy()

        #verif debug
        print(f"meilleur_patch is None : {meilleur_patch is None}")
        print(f"pixels_connus sum : {pixels_connus.sum()}")
        print(f"indices_patch_mask sum : {indices_patch_mask.sum()}")
        print(f"zone_interdite pixels bloqués : {np.sum(zone_interdite > 0)} / {h*w}")
        print(f"pix_coord : {pix_coord}")

        ys, xs = np.where(mask[ymin:ymax, xmin:xmax] > 0)
        print(f"pixels à remplir (ys) : {len(ys)}")
        
        ys_abs, xs_abs = ys + ymin, xs + xmin
        image[ys_abs, xs_abs]     = meilleur_patch[ys, xs]
        C[ys_abs, xs_abs] = C_p
        mask[ys_abs, xs_abs] = 0
        
        
        mask[ys_abs, xs_abs] = 0
        print(f"mask après maj : {np.sum(mask > 0)}")
        print(f"C après maj : {np.sum(C < 1.0)}")
        
    return image[R:R+h_orig, R:R+w_orig]

In [ ]:
resultat = algo_criminisi(img_m_rgb, mask, largeur_patch=21)
px.imshow(img_m_rgb).update_layout(title="Image depart").show()
px.imshow(resultat).update_layout(title="Image réparée").show()

Pixels restants : 17080
Nombre de contours : 1
  Contour 0 : 720 points, premier point : [[ 88 217]]
meilleur_patch is None : False
pixels_connus sum : 320
indices_patch_mask sum : 121
zone_interdite pixels bloqués : 24746 / 372400
pix_coord : [np.int32(52), np.int32(243)]
pixels à remplir (ys) : 121
mask après maj : 16959
C après maj : 17080
Pixels restants : 16959
Nombre de contours : 1
  Contour 0 : 719 points, premier point : [[ 88 217]]
meilleur_patch is None : False
pixels_connus sum : 320
indices_patch_mask sum : 121
zone_interdite pixels bloqués : 24625 / 372400
pix_coord : [np.int32(63), np.int32(243)]
pixels à remplir (ys) : 121
mask après maj : 16838
C après maj : 17080
Pixels restants : 16838
Nombre de contours : 1
  Contour 0 : 719 points, premier point : [[ 88 217]]
meilleur_patch is None : False
pixels_connus sum : 290
indices_patch_mask sum : 151
zone_interdite pixels bloqués : 24504 / 372400
pix_coord : [np.int32(74), np.int32(243)]
pixels à remplir (ys) : 151
mask apr

NameError: name 'cv2' is not defined